Дані, розбиття, ознаки, XGBoost

In [1]:
import os, re, glob, time
import numpy as np, pandas as pd
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

os.makedirs("results", exist_ok=True)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"не знайшов {name}: перевір Add Input"
    return hits[0]

tr = pd.read_csv(find("train_transaction.csv"))
idn = pd.read_csv(find("train_identity.csv"))
for c in tr.select_dtypes("float64").columns:
    tr[c] = tr[c].astype("float32")
df = tr.merge(idn, on="TransactionID", how="left")
df["has_identity"] = df["TransactionID"].isin(idn["TransactionID"])
df["day"] = (df["TransactionDT"] // 86400).astype(int)
df["week"] = df["day"] // 7
del tr

N = len(df)
TRAIN_W, VAL_W, TEST_W = (0, 17), (18, 21), (22, 26)
inw = lambda r: ((df.week >= r[0]) & (df.week <= r[1])).values
m_tr, m_va, m_te = inw(TRAIN_W), inw(VAL_W), inw(TEST_W)
assert (m_tr | m_va | m_te).all() and not (m_tr & m_va).any() and not (m_va & m_te).any()
assert (m_tr.sum(), m_va.sum(), m_te.sum()) == (431125, 77090, 82325)

ts, tid = df.TransactionDT.values, df.TransactionID.values
order = np.lexsort((tid, ts))
rank = np.empty(N, dtype=np.int64); rank[order] = np.arange(N)
y = df.isFraud.values
assert y.sum() == 20663
c1v = df["card1"].values

# ознаки таблиці (D1n додаємо ПІСЛЯ цього, щоб воно не потрапило в X)
X = df.drop(columns=["TransactionID", "isFraud", "TransactionDT", "day", "week"]).copy()
X["hour"] = (df.TransactionDT // 3600) % 24
X["weekday"] = (df.TransactionDT // 86400) % 7
X["has_identity"] = X["has_identity"].astype("int8")
for c in X.select_dtypes(include=["object", "category"]).columns:
    codes, _ = pd.factorize(X[c])
    X[c] = np.where(codes == -1, np.nan, codes).astype("float32")
df["D1n"] = (df["D1"] - df["TransactionDT"] / 86400).round()
spw = (y[m_tr] == 0).sum() / (y[m_tr] == 1).sum()

def metrics(y_true, p):
    k = int(y_true.sum()); o = np.argsort(-p)
    top1 = o[: max(1, int(0.01 * len(y_true)))]
    return {"pr_auc": average_precision_score(y_true, p),
            "roc_auc": roc_auc_score(y_true, p),
            "recall@k": y_true[o[:k]].sum() / k,
            "precision@1%": y_true[top1].mean()}

major = int(xgb.__version__.split(".")[0])
DEV = dict(tree_method="hist", device="cuda") if major >= 2 else dict(tree_method="gpu_hist")
print("готово:", X.shape, "| xgboost", xgb.__version__)

готово: (590540, 434) | xgboost 3.4.1


Мітки сусідів тієї ж картки (u2, u3, u5) з перевіркою

In [2]:
ALLK = {"u2": ["card1", "addr1"], "u3": ["card1", "addr1", "P_emaildomain"],
        "u5": ["card1", "addr1", "P_emaildomain", "D1n"]}

def label_feats(cols, W_days, DLd):
    valid = df[cols].notna().all(axis=1).values; idx = np.flatnonzero(valid)
    nk = np.full(N, np.nan, "float32"); fk = np.full(N, np.nan, "float32")
    for pos in df.loc[valid, cols].groupby(cols, sort=False).indices.values():
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt, ff = ts[rows], y[rows]; cs = np.concatenate([[0], np.cumsum(ff)])
        a = np.searchsorted(tt, tt, side="left")
        hi = np.minimum(np.searchsorted(tt, tt - DLd * 86400, side="right"), a)
        lo = np.minimum(np.searchsorted(tt, tt - W_days * 86400, side="left"), hi)
        nk[rows] = hi - lo; fk[rows] = cs[hi] - cs[lo]
    return nk, fk

_c = {}
def lab_cols(rel, W=30, dl=7):
    if (rel, W, dl) not in _c:
        nk, fk = label_feats(ALLK[rel], W, dl)
        _c[(rel, W, dl)] = pd.DataFrame({f"{rel}_nk": nk, f"{rel}_fk": fk,
            f"{rel}_rate": (fk / np.where(nk > 0, nk, np.nan)).astype("float32")})
    return _c[(rel, W, dl)]

def check_rel(rel, W=30, dl=7, n=60, seed=0):
    cols = ALLK[rel]; L = lab_cols(rel, W, dl); rng = np.random.default_rng(seed); bad = 0
    cand = np.flatnonzero(df[cols].notna().all(axis=1).values)
    for i in rng.choice(cand, n, replace=False):
        same = np.ones(N, bool)
        for c in cols: same &= (df[c].values == df[c].values[i])
        kn = same & (ts <= ts[i] - dl * 86400) & (ts >= ts[i] - W * 86400)
        if (int(kn.sum()) != int(L[f"{rel}_nk"].values[i]) or
                int(y[kn].sum()) != int(L[f"{rel}_fk"].values[i])): bad += 1
    return bad

for rel in ALLK:
    b = check_rel(rel); print(f"{rel}: розбіжностей {b}", flush=True); assert b == 0
BASE = pd.concat([X, lab_cols("u2"), lab_cols("u3"), lab_cols("u5")], axis=1)
print("база готова:", BASE.shape)

u2: розбіжностей 0
u3: розбіжностей 0
u5: розбіжностей 0
база готова: (590540, 443)


Верхня межа (скільки фроду взагалі лежить у рядках з відбитком)

In [3]:
FP = ["DeviceInfo", "id_30", "id_31", "id_33"]
fp_has = df[FP].notna().all(axis=1).values
for name, m in [("val", m_va), ("test", m_te)]:
    s = fp_has[m]
    print(f"{name}: рядків з відбитком {s.mean()*100:.1f} %, фроду там {int(y[m][s].sum())} "
          f"із {int(y[m].sum())} ({y[m][s].sum()/y[m].sum()*100:.1f} %), "
          f"частка фроду в цих рядках {y[m][s].mean()*100:.2f} %")

val: рядків з відбитком 7.5 %, фроду там 296 із 2645 (11.2 %), частка фроду в цих рядках 5.15 %
test: рядків з відбитком 9.9 %, фроду там 381 із 2868 (13.3 %), частка фроду в цих рядках 4.69 %


Двокрокова ознака + перевірка «в лоб»

In [4]:
card_ts, card_cs = {}, {}
for c, pos in df.groupby("card1", sort=False).indices.items():
    rows = pos[np.argsort(rank[pos], kind="stable")]
    card_ts[c] = ts[rows]; card_cs[c] = np.concatenate([[0], np.cumsum(y[rows])])

def two_hop(W_days=30, DLd=7):
    nk2 = np.full(N, np.nan, "float32"); fk2 = np.full(N, np.nan, "float32")
    nc2 = np.full(N, np.nan, "float32")
    idx_fp = np.flatnonzero(fp_has)
    for pos in df.loc[fp_has, FP].groupby(FP, sort=False).indices.values():
        rows = idx_fp[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt, cc = ts[rows], c1v[rows]
        for a in range(len(rows)):
            t_i = tt[a]
            lo = np.searchsorted(tt, t_i - W_days * 86400, side="left")
            others = set(cc[lo:a]); others.discard(cc[a])        # ІНШІ картки на цьому відбитку
            n = f = 0
            for c in others:
                T, CS = card_ts[c], card_cs[c]
                hi = np.searchsorted(T, t_i - DLd * 86400, side="right")
                l = np.searchsorted(T, t_i - W_days * 86400, side="left")
                if hi > l: n += hi - l; f += CS[hi] - CS[l]
            nk2[rows[a]] = n; fk2[rows[a]] = f; nc2[rows[a]] = len(others)
    return pd.DataFrame({"x2_nk": nk2, "x2_fk": fk2, "x2_ncards": nc2,
                         "x2_rate": (fk2 / np.where(nk2 > 0, nk2, np.nan)).astype("float32")})

def brute(i, W=30, DLd=7):
    same = np.ones(N, bool)
    for c in FP: same &= (df[c].values == df[c].values[i])
    link = same & (rank < rank[i]) & (ts >= ts[i] - W * 86400)
    S = set(c1v[link]); S.discard(c1v[i])
    if not S: return 0, 0, 0
    kn = np.isin(c1v, list(S)) & (ts <= ts[i] - DLd * 86400) & (ts >= ts[i] - W * 86400)
    return int(kn.sum()), int(y[kn].sum()), len(S)

t0 = time.time(); X2 = two_hop(); print("двокрокові ознаки:", round(time.time() - t0), "с", flush=True)
rng = np.random.default_rng(3); bad = 0
for i in rng.choice(np.flatnonzero(fp_has), 80, replace=False):
    if brute(i) != (int(X2["x2_nk"].values[i]), int(X2["x2_fk"].values[i]), int(X2["x2_ncards"].values[i])):
        bad += 1
print("розбіжностей з перерахунком «в лоб»:", bad); assert bad == 0
print("перевірки на витік пройдено")
print("рядків з відомим фродом на інших картках відбитка, %:",
      round(((X2["x2_fk"] > 0) & fp_has).sum() / fp_has.sum() * 100, 1), "від рядків з відбитком")

двокрокові ознаки: 30 с
розбіжностей з перерахунком «в лоб»: 0
перевірки на витік пройдено
рядків з відомим фродом на інших картках відбитка, %: 81.3 від рядків з відбитком


XGBoost, 5 seed

In [5]:
def run_fp(Xf, seed):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=6, subsample=0.8,
                      colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="aucpr",
                      early_stopping_rounds=50, random_state=seed, **DEV)
    m.fit(Xf[m_tr], y[m_tr], eval_set=[(Xf[m_va], y[m_va])], verbose=False)
    pv, pt = m.predict_proba(Xf[m_va])[:, 1], m.predict_proba(Xf[m_te])[:, 1]
    sv, st = fp_has[m_va], fp_has[m_te]
    r = metrics(y[m_te], pt)
    r.update(seed=seed, val_all=average_precision_score(y[m_va], pv),
             val_fp=average_precision_score(y[m_va][sv], pv[sv]),
             test_fp=average_precision_score(y[m_te][st], pt[st]))
    return r

VARS = {"база (u2+u3+u5)": BASE, "+ двокрокова (fp)": pd.concat([BASE, X2], axis=1)}
rows = []
for name, Xf in VARS.items():
    t0 = time.time()
    for sd in range(5):
        r = run_fp(Xf, sd); r["variant"] = name; rows.append(r)
    print(name, "готово,", round(time.time() - t0), "с", flush=True)
    pd.DataFrame(rows).to_csv("results/ieee_cis_two_hop_5seeds.csv", index=False)

R = pd.DataFrame(rows); b = "база (u2+u3+u5)"
P = lambda c: R.pivot(index="seed", columns="variant", values=c)[list(VARS)]
pt, pva, pvf, ptf = P("pr_auc"), P("val_all"), P("val_fp"), P("test_fp")
S = pd.DataFrame({"val PR-AUC": pva.mean(), "test PR-AUC": pt.mean(), "test std": pt.std(),
                  "Δtest (усі)": pt.sub(pt[b], axis=0).mean(),
                  "val, рядки з відбитком": pvf.mean(), "Δval (fp)": pvf.sub(pvf[b], axis=0).mean(),
                  "test, рядки з відбитком": ptf.mean(), "Δtest (fp)": ptf.sub(ptf[b], axis=0).mean(),
                  "std Δtest (fp)": ptf.sub(ptf[b], axis=0).std()}).round(3)
S["приймаємо (Δval fp ≥ +0,02 і Δtest fp > 0)"] = (S["Δval (fp)"] >= 0.02) & (S["Δtest (fp)"] > 0)
display(S); S.to_csv("results/ieee_cis_two_hop_summary.csv")

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [12:29:06] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


база (u2+u3+u5) готово, 116 с
+ двокрокова (fp) готово, 129 с


,val PR-AUC,test PR-AUC,test std,Δtest (усі),"val, рядки з відбитком",Δval (fp),"test, рядки з відбитком",Δtest (fp),std Δtest (fp),"приймаємо (Δval fp ≥ +0,02 і Δtest fp > 0)"
variant,,,,,,,,,,
база (u2+u3+u5),0.669,0.631,0.001,0.0,0.716,0.000,0.663,0.000,0.000,False
+ двокрокова (fp),0.668,0.631,0.002,-0.0,0.713,-0.003,0.661,-0.003,0.008,False
